# Sesi 13: SQL dengan Python

Di sesi ini kita belajar cara menjalankan **SQL langsung dari Python**
menggunakan library bawaan Python: 'sqlite3'

**Apa itu SQLite?**
- Database ringan yang tersimpan dalam satu file '.db'
- Tidak perlu install server terpisah
- Cocok untuk analisis data, prototyping, dan belajar SQL

**Yang akan kita pelajari:**
1. Membuat database SQLite dari CSV
2. Query dasar: SELECT, WHERE, ORDER BY, LIMIT
3. Agregasi: GROUP BY, COUNT, SUM, AVG
4. Filter lanjutan: HAVING, BETWEEN, LIKE
5. Membaca hasil query langsung ke Pandas DataFrame

In [2]:
import sqlite3
import pandas as pd

# Load dataset Superstore
df = pd.read_csv("samplesuperstore.csv", encoding="latin1")
print(f"Dataset shape: {df.shape}")
print(df.dtypes)

Dataset shape: (10194, 21)
Row ID              int64
Order ID           object
Order Date         object
Ship Date          object
Ship Mode          object
Customer ID        object
Customer Name      object
Segment            object
Country/Region     object
City               object
State/Province     object
Postal Code        object
Region             object
Product ID         object
Category           object
Sub-Category       object
Product Name       object
Sales             float64
Quantity            int64
Discount          float64
Profit            float64
dtype: object


## Langkah 1: Buat Database SQLite dari CSV

Kita simpan data Superstore ke dalam database SQLite agar bisa diquery pakai SQL

In [6]:
# Buat koneksi ke database (otomatis buat file .db baru)
conn = sqlite3.connect("superstore.db")

# Simpan DataFrame ke table SQLite bernama "superstore"
df.to_sql("superstore", conn, if_exists="replace", index=False)

print("Database berhasil dibuat")
print("Tabel 'superstore' sudah tersimpan di superstore.db")

Database berhasil dibuat
Tabel 'superstore' sudah tersimpan di superstore.db


## Langkah 2: Query Dasar - SELECT

Fungsi 'pd.read_sql()' memungkinkan kita menulis SQL hasilnya langsung jadi DataFrame

In [12]:
# Ambil 5 baris pertama
query = "SELECT * FROM superstore LIMIT 5"
result = pd.read_sql(query, conn)
result

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,1/3/2023,1/7/2023,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,1/4/2023,1/8/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,1/4/2023,1/8/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,1/4/2023,1/8/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,1/5/2023,1/12/2023,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


## Langkah 3: WHERE - Filter Data

In [16]:
# Tampilkan transaksi di kategori "Technology" dengan Profit > 500
query = """
SELECT "Order ID", "Product Name", "Category", Sales, Profit
FROM superstore
WHERE Category = 'Technology' AND Profit > 500
ORDER BY Profit DESC
LIMIT 10
"""
result = pd.read_sql(query, conn)
result

,Order ID,Product Name,Category,Sales,Profit
0,US-2025-118689,Canon imageCLASS 2200 Advanced Copier,Technology,17499.950,8399.9760
1,US-2026-140151,Canon imageCLASS 2200 Advanced Copier,Technology,13999.960,6719.9808
2,US-2026-166709,Canon imageCLASS 2200 Advanced Copier,Technology,10499.970,5039.9856
3,US-2026-127180,Canon imageCLASS 2200 Advanced Copier,Technology,11199.968,3919.9888
4,US-2025-158841,HP Designjet T520 Inkjet Large Format Printer ...,Technology,8749.950,2799.9840
5,US-2025-140158,Hewlett Packard LaserJet 3310 Copier,Technology,5399.910,2591.9568
6,US-2025-143819,Ativa V4110MDD Micro-Cut Shredder,Technology,4899.930,2400.9657
7,US-2025-107440,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,9099.930,2365.9818
8,US-2024-128587,Canon PC1060 Personal Laser Copier,Technology,4899.930,2302.9671
9,US-2023-145541,HP Designjet T520 Inkjet Large Format Printer ...,Technology,6999.960,2239.9872


## Langkah 4: GROUP BY - Agregasi per Kategori

In [18]:
# Total Sales & Profit per Kategori
query = """
SELECT
    Category,
    COUNT(*) AS jumlah_transaksi,
    ROUND(SUM(Sales), 2) AS total_sales,
    ROUND(SUM(Profit), 2) AS total_profit,
    ROUND(AVG(Discount), 2) AS rata_diskon
FROM superstore
GROUP BY Category
ORDER BY total_profit DESC
"""
result = pd.read_sql(query, conn)
result

,Category,jumlah_transaksi,total_sales,total_profit,rata_diskon
0,Technology,1865,839893.28,146543.38,0.13
1,Office Supplies,6128,731893.31,126023.44,0.16
2,Furniture,2201,754747.76,19730.00,0.17


## Langkah 5: HAVING - Filter Setelah Agregasi

'HAVING'dipakai untuk filter hasil GROUP BY (bukan baris individual seperti WHERE)

In [24]:
# Sub-kategori dengan total profit NEGATIF (merugi)
query = """
SELECT
    "Sub-Category",
    ROUND(SUM(Profit), 2) AS total_profit,
    COUNT(*) AS jumlah_transaksi
FROM superstore
GROUP BY "Sub-Category"
HAVING total_profit < 0
ORDER BY total_profit ASC
"""
result = pd.read_sql(query, conn)
result

,Sub-Category,total_profit,jumlah_transaksi
0,Tables,-17753.21,326
1,Bookcases,-3632.07,232
2,Supplies,-1171.39,192


## Kesimpulan sesi 13

- 'sqlite3' memungkinkan kita membuat dan query database langsung dari Python
- 'pd.read_sql' menggabungkan kekuatan SQL + Pandas dalam satu baris
- SQL sangat berguna untuk filter, agregasi, dan analisis data besar
- Temuan: beberapa Sub-Category seperti **Tables** dan **Bookcases** memiliki total profit negatif - konsisten dengan temuan di sesi EDA sebelumnya

**Next:** Sesi 14 - Dashboard Interaktif dengan Plotly / Streamlit

In [26]:
# Selalu tutup koneksi setelah selesai
conn.close()
print("Koneksi database ditutup")

Koneksi database ditutup
